#**Data Cleaning**

##**Load Data**

In [3]:
from google.colab import drive
import pandas as pd
import numpy as np
import json
import re
import os
import time

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

folder_path = '/content/drive/MyDrive/HPDP_Project1/data'
raw_file = os.path.join(folder_path, 'raw_data.json')
clean_file = os.path.join(folder_path, 'cleaned_data.csv')

print("Checking for data file...")

if os.path.exists(raw_file):
    file_size_mb = os.path.getsize(raw_file) / (1024 * 1024)
    print("Success! Found the raw data file.")
    print(f"File Path: {raw_file}")
    print(f"File Size: {file_size_mb:.2f} MB")
else:
    print("ERROR: Cannot find 'raw_data.json'.")


Mounted at /content/drive
Checking for data file...
Success! Found the raw data file.
File Path: /content/drive/MyDrive/HPDP_Project1/data/raw_data.json
File Size: 59.63 MB


In [2]:
print("Reading raw data file... please wait.")
df_raw = pd.read_json(raw_file)

print("\n==============================================")
print("RAW DATASET INITIAL SUMMARY")
print("==============================================")
print(f"Total rows collected: {len(df_raw):,}")

print("\nPreviewing the First 5 Rows:")
display(df_raw.head(5))

print("\nColumn Names and Data Types:")
print(df_raw.dtypes)

print("\nChecking for Missing/Null Values:")
print(df_raw.isnull().sum())

Reading raw data file... please wait.

RAW DATASET INITIAL SUMMARY
Total rows collected: 120,100

Previewing the First 5 Rows:


,job_title,occupation_category,salary_range,contract_type,working_hours,education_level,location,employer_name,job_url
0,Clerk Assistant (Ref: #132886),Administrative & Support Services,RM 2530 - RM 3946,Part-Time,Normal Hours,Bachelor's Degree or Equivalent,Pahang,N/A,https://myfuturejobs.gov.my/administrative-and...
1,Office Executive (Ref: #327732),Administrative & Support Services,RM 2791 - RM 5885,Permanent,2 Shift Time,SPM / O Level / SKM Level 1 / SKM Level 2 / SK...,Johor,N/A,https://myfuturejobs.gov.my/administrative-and...
2,Office Executive (Ref: #511550),Administrative & Support Services,RM 2211 - RM 6197,Internship,Normal Hours,SPM / O Level / SKM Level 1 / SKM Level 2 / SK...,Penang,N/A,https://myfuturejobs.gov.my/administrative-and...
3,Office Executive (Ref: #366654),Administrative & Support Services,RM 2273 - RM 6179,Permanent,2 Shift Time,SPM / O Level / SKM Level 1 / SKM Level 2 / SK...,Perak,N/A,https://myfuturejobs.gov.my/administrative-and...
4,Clerk Assistant (Ref: #768107),Administrative & Support Services,RM 2485 - RM 4278,Part-Time,Full-Time,SPM / O Level / SKM Level 1 / SKM Level 2 / SK...,Penang,N/A,https://myfuturejobs.gov.my/administrative-and...



Column Names and Data Types:
job_title              object
occupation_category    object
salary_range           object
contract_type          object
working_hours          object
education_level        object
location               object
employer_name          object
job_url                object
dtype: object

Checking for Missing/Null Values:
job_title              0
occupation_category    0
salary_range           0
contract_type          0
working_hours          0
education_level        0
location               0
employer_name          0
job_url                0
dtype: int64


# **Cleaning Job Title**

In [4]:
# Load the raw JSON file into a Pandas DataFrame
try:
    df = pd.read_json(raw_file)
    print(f"Successfully loaded {len(df)} raw entries from the web crawler.")
except Exception as e:
    print(f"Error loading file: {e}. Ensure 'raw_data.json' is in your Drive.")

# Clean Job Titles & Isolate the Vacancy Reference ID
# Example: "Clerk Assistant (Ref: #132886)" -> Title: "Clerk Assistant", ID: "132886"
def clean_title_and_extract_id(row):
    title_raw = str(row['job_title'])
    match = re.search(r'(.*?)\s*\(Ref:\s*#(\d+)\)', title_raw)
    if match:
        return match.group(1).strip(), match.group(2)
    return title_raw, np.nan

print("\nProcessing job titles and extracting reference tracking keys...")
df['job_title'], df['vacancy_id'] = zip(*df.apply(clean_title_and_extract_id, axis=1))

print("Job titles cleaned and 'vacancy_id' column created successfully!")
print("\nPreview of processed titles and IDs:")
display(df[['job_title', 'vacancy_id']].head(5))

Successfully loaded 120100 raw entries from the web crawler.

Processing job titles and extracting reference tracking keys...
Job titles cleaned and 'vacancy_id' column created successfully!

Preview of processed titles and IDs:


,job_title,vacancy_id
0,Clerk Assistant,132886
1,Office Executive,327732
2,Office Executive,511550
3,Office Executive,366654
4,Clerk Assistant,768107


# **Cleaning Salary Column**

In [ ]:
# Parse Text Salary Ranges into Separate Numerical Columns
# Example: "RM 2530 - RM 3946" -> Min: 2530.0, Max: 3946.0, Avg: 3238.0
def convert_salary_to_numbers(salary_str):
    if pd.isna(salary_str) or 'N/A' in str(salary_str):
        return np.nan, np.nan
    numbers = [float(x.replace(',', '')) for x in re.findall(r'\d[\d,]*', str(salary_str))]
    if len(numbers) >= 2:
        return numbers[0], numbers[1]
    elif len(numbers) == 1:
        return numbers[0], numbers[0]
    return np.nan, np.nan

print("Vectorizing text salary fields into structured numeric data streams...")
df['salary_min'], df['salary_max'] = zip(*df['salary_range'].apply(convert_salary_to_numbers))
df['salary_average'] = (df['salary_min'] + df['salary_max']) / 2

print("Salary fields successfully converted to numeric columns!")
print("\nPreview of new numerical salary tracking streams:")
display(df[['job_title', 'salary_min', 'salary_max', 'salary_average']].head(5))

Vectorizing text salary fields into structured numeric data streams...
Salary fields successfully converted to numeric columns!

Preview of new numerical salary tracking streams:


,job_title,salary_min,salary_max,salary_average
0,Clerk Assistant,2530.0,3946.0,3238.0
1,Office Executive,2791.0,5885.0,4338.0
2,Office Executive,2211.0,6197.0,4204.0
3,Office Executive,2273.0,6179.0,4226.0
4,Clerk Assistant,2485.0,4278.0,3381.5


#**Remove duplicate**

In [ ]:
initial_row_count = len(df)
df.drop_duplicates(subset=['vacancy_id'], keep='first', inplace=True)
final_row_count = len(df)

print("==============================================")
print("DEDUPLICATION PURGE REPORT")
print("==============================================")
print(f"Starting Row Count : {initial_row_count:,}")
print(f"Rows Safely Deleted: {initial_row_count - final_row_count:,}")
print(f"Remaining Unique Rows  : {final_row_count:,}")
print("==============================================")

DEDUPLICATION PURGE REPORT
Starting Row Count : 120,100
Rows Safely Deleted: 7,601
Remaining Unique Rows  : 112,499


#**Handling missing fields and drop unused columns**

In [ ]:
print("Standardizing missing fields and column drops...")

pd.set_option('future.no_silent_downcasting', True)

df.replace("N/A", np.nan, inplace=True)

df.drop(columns=['employer_name'], inplace=True, errors='ignore')

print("Missing fields standardized and unused columns dropped.")
print("\nFinal check for Null/NaN values across our remaining columns:")
print(df.isnull().sum())

Standardizing missing fields and column drops...
Missing fields standardized and unused columns dropped.

Final check for Null/NaN values across our remaining columns:
job_title              0
occupation_category    0
salary_range           0
contract_type          0
working_hours          0
education_level        0
location               0
job_url                0
vacancy_id             0
salary_min             0
salary_max             0
salary_average         0
dtype: int64


#**Export clean dataset**

In [ ]:
os.makedirs(os.path.dirname(clean_file), exist_ok=True)
df.to_csv(clean_file, index=False)

print("\n--- PIPELINE RUN COMPLETE ---")
print("==============================================")
print("FINAL VALIDATED DATA SAMPLE (TOP 5 ROWS):")
print("==============================================")
display(df[['job_title', 'vacancy_id', 'salary_min', 'salary_max', 'salary_average']].head(5))
print("==============================================")
print(f"Clean data successfully exported at:\n{clean_file}")


--- PIPELINE RUN COMPLETE ---
FINAL VALIDATED DATA SAMPLE (TOP 5 ROWS):


,job_title,vacancy_id,salary_min,salary_max,salary_average
0,Clerk Assistant,132886,2530.0,3946.0,3238.0
1,Office Executive,327732,2791.0,5885.0,4338.0
2,Office Executive,511550,2211.0,6197.0,4204.0
3,Office Executive,366654,2273.0,6179.0,4226.0
4,Clerk Assistant,768107,2485.0,4278.0,3381.5


Clean data successfully exported at:
/content/drive/MyDrive/HPDP_Project1/data/cleaned_data.csv
